# Benchmarks

## Initialize

In [1]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [2]:
import ray
ray.shutdown()

In [3]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [4]:
import ray
ray.init(num_cpus=24, include_dashboard=False)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

{'node_ip_address': '10.32.105.2',
 'raylet_ip_address': '10.32.105.2',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-20_10-58-45_407952_324559/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-20_10-58-45_407952_324559/sockets/raylet',
 'webui_url': None,
 'session_dir': '/tmp/ray/session_2022-04-20_10-58-45_407952_324559',
 'metrics_export_port': 44177,
 'gcs_address': '10.32.105.2:48923',
 'address': '10.32.105.2:48923',
 'node_id': '4d8f11f81450be439fc7ec1f3180b0d2eae2b20998801bdce0501f1a'}

In [5]:
endpoints = sorted(pd.read_csv(f"{experiment_path}/endpoints.txt", header=None)[0].to_list())

In [6]:
in_path = pathlib.Path(f"{experiment_path}/loghs")
in_path.mkdir(parents=True, exist_ok=True)

out_path = f"{experiment_path}/coxph/input"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [7]:
models = models = [f.name for f in in_path.iterdir() if f.is_dir() and "ipynb_checkpoints" not in str(f)]
partitions = [i for i in range(22)] #[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [29]:
from sklearn.preprocessing import StandardScaler
import pickle
import zstandard

def read_data(fp_in, split):
    temp = pd.read_feather(f"{fp_in}/{split}.feather").set_index("eid")
    return temp   
    
def save_pickle(data, data_path):
    with open(data_path, "wb") as fh:
        cctx = zstandard.ZstdCompressor()
        with cctx.stream_writer(fh) as compressor:
            compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))
    
def read_predictions(model, partition, split):

    fp_in = f"{in_path}/{model}/{partition}"
    
    if pathlib.Path(fp_in).is_dir(): temp = read_data(fp_in, split)
    return temp

In [18]:
models

['Identity(AgeSex+Records)+MLP', 'Identity(Records)+MLP']

In [31]:
for partition in partitions:
    for split in ["train", "valid", "test"]:
        temp = read_predictions('Identity(Records)+MLP', partition)
        print(partition, split, (temp.isna().sum() > 0).sum())

0 train 0
0 valid 0
0 test 0
1 train 0
1 valid 0
1 test 0
2 train 0
2 valid 0
2 test 0
3 train 0
3 valid 0
3 test 0
4 train 0
4 valid 0
4 test 0
5 train 0
5 valid 0
5 test 0
6 train 0
6 valid 0
6 test 0
7 train 0
7 valid 0
7 test 0
8 train 0
8 valid 0
8 test 0
9 train 0
9 valid 0
9 test 0
10 train 0
10 valid 0
10 test 0
11 train 0
11 valid 0
11 test 0
12 train 0
12 valid 0
12 test 0
13 train 0
13 valid 0
13 test 0
14 train 0
14 valid 0
14 test 0
15 train 0
15 valid 0
15 test 0
16 train 0
16 valid 0
16 test 0
17 train 0
17 valid 0
17 test 0
18 train 0
18 valid 0
18 test 0
19 train 0
19 valid 0
19 test 0
20 train 0
20 valid 0
20 test 0
21 train 0
21 valid 0
21 test 0
